# PEMS-SF 数据分组预处理 (train/test)
职责：从原始 data/ 解析每日矩阵与标签，按 Stage5 的 G1-G5 分组生成站点掩码，对齐 stations_list 顺序，并导出轻量分组产物供训练使用。
说明：不做可视化与模型，专注分组与一致性校验，避免与 v3.5 重复。

In [ ]:
# 依赖与路径
import pathlib, json, re
import numpy as np
import pandas as pd
ROOT = pathlib.Path('.')
DATA_DIR = ROOT / 'data'
STAGE5_DIR = ROOT / 'Stage5-figure'
OUT_DIR = DATA_DIR / 'processed'
OUT_DIR.mkdir(parents=True, exist_ok=True)
print('数据目录:', DATA_DIR.resolve())

In [ ]:
# 基础解析：与 v3.5 一致的每日矩阵解析
def parse_day_matrix(line: str, expected_rows=963, expected_cols=144):
    line = line.strip()
    if line.startswith('[') and line.endswith(']'): line = line[1:-1]
    rows = [r.strip() for r in line.split(';') if r.strip()]
    data = []
    for r in rows:
        nums = [float(x) for x in r.split() if re.match(r'^-?\d+(\.\d+)?$', x)]
        if nums: data.append(nums)
    if not data:
        return None
    max_len = max(len(rr) for rr in data)
    arr = np.full((len(data), max_len), np.nan, dtype=float)
    for i, rr in enumerate(data): arr[i, :len(rr)] = rr
    return arr

def iter_day_matrices(path: pathlib.Path, limit=None):
    with path.open('r', encoding='utf-8', errors='ignore') as f:
        for i, line in enumerate(f):
            if limit is not None and i >= limit: break
            mat = parse_day_matrix(line)
            if mat is None: continue
            yield mat

def load_labels(path: pathlib.Path):
    txt = path.read_text(encoding='utf-8', errors='ignore').strip().strip('[]')
    return np.array([int(x) for x in txt.split() if re.match(r'^\d+$', x)], dtype=int)

In [ ]:
# 读取基础文件
train_path = DATA_DIR / 'PEMS_train'
test_path  = DATA_DIR / 'PEMS_test'
labels_train = load_labels(DATA_DIR / 'PEMS_trainlabels')
labels_test  = load_labels(DATA_DIR / 'PEMS_testlabels')
station_ids_txt = (DATA_DIR / 'stations_list').read_text(encoding='utf-8', errors='ignore').strip().strip('[]')
station_ids = [int(x) for x in station_ids_txt.split() if re.match(r'^\d+$', x)]
assert len(station_ids) == 963, f'stations_list 数量异常: {len(station_ids)}'
print('Train days:', len(labels_train), 'Test days:', len(labels_test), 'Stations:', len(station_ids))

In [ ]:
# 加载 Stage5 的 G1..G5 分组并生成掩码（对齐 stations_list 顺序）
groups_index_path = STAGE5_DIR / 'groups_g1_g5.json'
if not groups_index_path.exists():
    raise FileNotFoundError('缺少 Stage5-figure/groups_g1_g5.json，请先运行 v3.5 聚类分组。')
groups_index = json.loads(groups_index_path.read_text(encoding='utf-8'))
group_masks = {}
for g in ['g1','g2','g3','g4','g5']:
    sids = set(groups_index.get(g, []))
    mask = np.array([sid in sids for sid in station_ids], dtype=bool)
    group_masks[g] = mask
    print(g, 'stations:', int(mask.sum()))

In [ ]:
# 审计：快速检查前若干天形状一致性与掩码适配情况
def audit_split(path: pathlib.Path, expect_shape=(963,144), limit=20):
    ok, bad = 0, []
    for i, mat in enumerate(iter_day_matrices(path, limit=limit)):
        if mat.shape == expect_shape: ok += 1
        else: bad.append((i, mat.shape))
    return {'iterated': i+1 if 'i' in locals() else 0, 'ok': ok, 'bad': bad}
audit_train = audit_split(train_path, limit=30)
audit_test  = audit_split(test_path,  limit=30)
print('Train preview:', audit_train)
print('Test  preview:', audit_test)

In [ ]:
# 导出分组产物：仅导出站点掩码、索引与基本统计，供训练直接加载
export = {
    'station_ids': station_ids,
    'groups': {k: np.where(v)[0].tolist() for k, v in group_masks.items()},
    'groups_masks_bool': {k: v.astype(bool).tolist() for k, v in group_masks.items()},
    'labels': {
        'train': labels_train.tolist(),
        'test': labels_test.tolist()
    },
    'meta': {
        'train_days': len(labels_train),
        'test_days': len(labels_test),
        'stations': len(station_ids)
    }
}
out_json = OUT_DIR / 'pems_sf_groups_index.json'
out_npz  = OUT_DIR / 'pems_sf_groups_masks.npz'
out_json.write_text(json.dumps(export, ensure_ascii=False, indent=2), encoding='utf-8')
np.savez(out_npz, **{f'{k}_mask': v.astype(np.bool_) for k, v in group_masks.items()}, station_ids=np.array(station_ids, dtype=np.int32))
print('✓ 导出:', out_json)
print('✓ 导出:', out_npz)

## 使用说明（训练端）
- 读取 data/processed/pems_sf_groups_index.json 获取 station_ids 与分组索引。
- 或读取 data/processed/pems_sf_groups_masks.npz 获取布尔掩码，按 stations_list 顺序与每日矩阵对齐。
- 训练端可直接遍历每日矩阵，应用目标组掩码，构造样本与标签（与 v2.0.0 一致）。

### 训练端使用说明
- 读取 data/processed/groups_index.json 获取统一分组（g1..g5 -> station_ids）。
- 如需分别分析 train/test 的分组分布，可读取 groups_train.json 与 groups_test.json。
- 模型训练时按 groups_index 生成掩码，并对 train/test 的每日矩阵应用相同站点选择。

## 训练驱动的分组与测试映射 (方式A，严格对齐 v3.5 分组)
以 v3.5 生成的 Stage5-figure/groups_g1_g5.json 作为训练分组的唯一来源；不重新聚类。基于训练集站点的 5D 形状特征在同一标准化空间计算各组中心，并将测试集站点按最近中心分配至 g1..g5。

In [ ]:
import json, pathlib, numpy as np, pandas as pd
from scipy.signal import find_peaks
from sklearn.preprocessing import StandardScaler

DATA_DIR = pathlib.Path('data')
STAGE5_DIR = pathlib.Path('Stage5-figure')
OUT_DIR = DATA_DIR / 'processed'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# 加载基础文件与 v3.5 分组索引
def _load_stations_list(path: pathlib.Path):
    raw = path.read_text(encoding='utf-8', errors='ignore').strip().strip('[]')
    return [int(x) for x in raw.split() if x.isdigit()]

def _load_labels(path: pathlib.Path):
    txt = path.read_text(encoding='utf-8', errors='ignore').strip().strip('[]')
    return np.array([int(x) for x in txt.split() if x.isdigit()], dtype=int)

stations_list_path = DATA_DIR / 'stations_list'
station_ids = _load_stations_list(stations_list_path)
labels_train = _load_labels(DATA_DIR / 'PEMS_trainlabels')
labels_test  = _load_labels(DATA_DIR / 'PEMS_testlabels')
train_path = DATA_DIR / 'PEMS_train'
test_path  = DATA_DIR / 'PEMS_test'
assert len(station_ids) == 963, '站点数量应为 963'
groups_index_path = STAGE5_DIR / 'groups_g1_g5.json'
assert groups_index_path.exists(), '缺少 v3.5 分组文件 Stage5-figure/groups_g1_g5.json'
groups_index = json.loads(groups_index_path.read_text(encoding='utf-8'))
print('✓ 载入 v3.5 分组索引：', {k: len(v) for k,v in groups_index.items()})

# 与前文一致的日矩阵解析
def parse_day_matrix(line: str, expected_rows=963, expected_cols=144):
    line = line.strip()
    if line.startswith('[') and line.endswith(']'): line = line[1:-1]
    rows = [r.strip() for r in line.split(';') if r.strip()]
    data = []
    for r in rows:
        nums = [float(x) for x in r.split() if x.replace('.', '', 1).isdigit()]
        if nums: data.append(nums)
    if not data: return None
    arr = np.full((len(data), max(len(rr) for rr in data)), np.nan, dtype=float)
    for i, rr in enumerate(data): arr[i, :len(rr)] = rr
    return arr

def iter_day_matrices(path: pathlib.Path, limit=None):
    with path.open('r', encoding='utf-8', errors='ignore') as f:
        for i, line in enumerate(f):
            if limit is not None and i >= limit: break
            mat = parse_day_matrix(line)
            if mat is None: continue
            yield mat

# 构造 station × 144 的代表曲线（沿天求均值）
def build_station_representative_curve(split_path: pathlib.Path, labels: np.ndarray):
    sensors, slots = 963, 144
    acc_sum = np.zeros((sensors, slots), dtype=np.float32)
    acc_cnt = np.zeros((sensors,), dtype=np.int32)
    for day_i, mat in enumerate(iter_day_matrices(split_path)):
        if day_i >= len(labels): break
        if mat.shape[0] != sensors or mat.shape[1] < slots:
            continue
        acc_sum += mat[:, :slots].astype(np.float32)
        acc_cnt += 1
    acc_cnt_safe = np.where(acc_cnt==0, 1, acc_cnt)
    rep = acc_sum / acc_cnt_safe[:, None]
    rep = np.nan_to_num(rep, nan=0.0, posinf=1.0, neginf=0.0)
    rep = np.clip(rep, 0.0, 1.0)
    return rep

def extract_5d_features_from_curve(curve144: np.ndarray):
    peaks, _ = find_peaks(curve144, prominence=0.03, width=2)
    n_peaks = min(len(peaks), 3) / 3.0
    denom = float(np.max(curve144)) + 1e-8
    am_peak = float(np.max(curve144[36:55])) / denom
    pm_peak = float(np.max(curve144[96:115])) / denom
    wd_we_var = float(np.linalg.norm(curve144[36:60] - curve144[96:120]) / np.sqrt(24))
    wd_we_var = float(np.clip(wd_we_var, 0, 1))
    weekend_anom = float(np.clip(np.mean(curve144[30:55]) / 0.5, 0, 1))
    return np.array([n_peaks, am_peak, pm_peak, wd_we_var, weekend_anom], dtype=np.float32)

def build_5d_features_for_split(split_path: pathlib.Path, labels: np.ndarray):
    rep = build_station_representative_curve(split_path, labels)  # (963,144)
    feats = np.zeros((rep.shape[0], 5), dtype=np.float32)
    for i in range(rep.shape[0]):
        feats[i] = extract_5d_features_from_curve(rep[i])
    return feats, rep

# 训练集特征与标准化（不聚类）
train_feats, train_rep = build_5d_features_for_split(train_path, labels_train)
scaler = StandardScaler().fit(train_feats)
X_train_std = scaler.transform(train_feats)

# 将 v3.5 分组映射到 stations_list 顺序的索引与掩码
group_idx_map = {g: set(groups_index.get(g, [])) for g in ['g1','g2','g3','g4','g5']}
group_masks = {g: np.array([sid in group_idx_map[g] for sid in station_ids], dtype=bool) for g in group_idx_map}
group_indices_by_order = {g: np.where(mask)[0] for g, mask in group_masks.items()}
print('✓ 对齐 stations_list 生成掩码:', {g: int(m.sum()) for g,m in group_masks.items()})

# 计算各组中心（在标准化空间内）
centroids_std = []
for g in ['g1','g2','g3','g4','g5']:
    idx = group_indices_by_order[g]
    if idx.size == 0:
        centroids_std.append(np.zeros((5,), dtype=np.float32))
    else:
        centroids_std.append(X_train_std[idx].mean(axis=0))
centroids_std = np.vstack(centroids_std)  # (5,5)

# 测试集特征并按最近中心分配
test_feats, test_rep = build_5d_features_for_split(test_path, labels_test)
X_test_std = scaler.transform(test_feats)
assign_test = []
for i in range(X_test_std.shape[0]):
    dists = np.linalg.norm(centroids_std - X_test_std[i], axis=1)
    gid = int(np.argmin(dists)) + 1  # 1..5
    assign_test.append(gid)
assign_test = np.array(assign_test, dtype=int)

# 组织分组结果：train 使用 v3.5 原分组；test 使用最近中心分配
groups_train = {g: [sid for sid in station_ids if sid in group_idx_map[g]] for g in ['g1','g2','g3','g4','g5']}
groups_test  = {f'g{gid}': [station_ids[i] for i in range(len(station_ids)) if assign_test[i]==gid] for gid in range(1,6)}

export = {
    'station_ids': station_ids,
    'groups_index': groups_train,  # 统一索引=训练分组(v3.5)
    'groups_train': groups_train,
    'groups_test': groups_test,
    'meta': {
        'train_days': int(len(labels_train)),
        'test_days': int(len(labels_test)),
        'stations': int(len(station_ids)),
        'method': 'Use v3.5 groups for train; nearest-centroid assign for test in 5D standardized space'
    }
}
(OUT_DIR / 'groups_train.json').write_text(json.dumps(groups_train, ensure_ascii=False, indent=2), encoding='utf-8')
(OUT_DIR / 'groups_test.json').write_text(json.dumps(groups_test, ensure_ascii=False, indent=2), encoding='utf-8')
(OUT_DIR / 'groups_index.json').write_text(json.dumps(groups_train, ensure_ascii=False, indent=2), encoding='utf-8')
(OUT_DIR / 'groups_export_all.json').write_text(json.dumps(export, ensure_ascii=False, indent=2), encoding='utf-8')
print('✓ 导出: ', (OUT_DIR / 'groups_index.json').resolve())
print('✓ 导出: ', (OUT_DIR / 'groups_train.json').resolve())
print('✓ 导出: ', (OUT_DIR / 'groups_test.json').resolve())